In [2]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

def create_manifest(
    data_root,
    output_csv = "manifest.csv",
    spatial_holdout_tiles = ["N00E039"], # Tile(s) strictly reserved for testing

):
    """
    Scans the dataset directory, applies split logic (with 14-day gaps), 
    constructs file paths, and saves a manifest CSV for the PyTorch DataLoader.
    """
    data_root = Path(data_root)
    records = []
    
    
    

In [13]:
data_root = "D:/flood_data"
data_root = Path(data_root)
records = []
spatial_holdout_tiles = ["N00E039"]

# 1. Find all flood date CSVs
csv_files = list(data_root.rglob("*-post-processing_flood_dates.csv"))
print(f"Found {len(csv_files)} flood date CSV files.")

# 2. Iterate and collect all events
for csv_path in csv_files:
    # Extract tile name from the file path
    tile = csv_path.parent.name

    df = pd.read_csv(csv_path)

    for date_str in df['date']:
        records.append({
            "tile": tile,
            "date": date_str
        })
        

df_manifest = pd.DataFrame(records)
df_manifest['date_obj'] = pd.to_datetime(df_manifest['date'], format="%Y-%m-%d")


# 3. Apply Temporal Splitting & The 14-Day Purge Gap
# We want Train: <=2021, Val: 2022, Test: >=2023.
# To prevent 14-day rolling rain features from overlapping, we add a 14-day deadzone.
train_end = datetime(2021, 12, 31) - timedelta(days=14)
val_start = datetime(2022, 1, 1)
val_end = datetime(2022, 12, 31) - timedelta(days=14)
test_start = datetime(2023, 1, 1)

def assign_split(row):
    # Rule 1: Spatial Holdout ALWAYS goes to Test
    if row['tile'] in spatial_holdout_tiles:
        return "test"
    
    # Rule 2: Temporal Logic (Chronological Generalization)
    d = row['date_obj']
    if d <= train_end:
        return "train"
    elif val_start <= d <= val_end:
        return "val"
    elif d >= test_start:
        return "test"
    else:
        return 'drop' # Falls inside the 14-day purge gap!
    
df_manifest['split'] = df_manifest.apply(assign_split, axis = 1)

# Drop the deadzone events
df_manifest = df_manifest[df_manifest['split'] != 'drop'].copy()


Found 12 flood date CSV files.


In [14]:
df_manifest.head()

,tile,date,date_obj,split
0,N00E033,2014-10-28,2014-10-28,train
1,N00E033,2014-11-07,2014-11-07,train
2,N00E033,2014-11-12,2014-11-12,train
3,N00E033,2014-11-21,2014-11-21,train
4,N00E033,2014-12-01,2014-12-01,train


In [12]:
pd.DataFrame(records)

,tile,date
0,N00E033,2014-10-28
1,N00E036,2014-11-14
2,N00E039,2014-11-28
3,N03E033,2014-11-21
4,N03E036,2014-10-16
5,N03E039,2014-10-28
6,S03E033,2014-10-28
7,S03E036,2014-11-14
8,S03E039,2014-11-14
9,S06E033,2014-11-21


In [9]:
df = pd.read_csv(csv_files[0])
df.head()

,date
0,2014-10-28
1,2014-11-07
2,2014-11-12
3,2014-11-21
4,2014-12-01


In [ ]:


import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

def create_manifest(
    data_root: str,
    output_csv: str = "manifest.csv",
    spatial_holdout_tiles: list = ["N00E039"], # Tile(s) strictly reserved for testing
    verify_paths: bool = False
):
    """
    Scans the dataset directory, applies split logic (with 14-day gaps), 
    constructs file paths, and saves a manifest CSV for the PyTorch DataLoader.
    """
    data_root = Path(data_root)
    records =[]
    
    # 1. Find all flood date CSVs (from your previous extraction step)
    csv_files = list(data_root.rglob("*-post-processing_flood_dates.csv"))
    print(f"Found {len(csv_files)} tile date CSVs.")

    # 2. Iterate and collect all events
    for csv_path in csv_files:
        # Extract tile name from the filename (e.g., 'N00E036' from 'N00E036-post...')
        tile = csv_path.parent.name 
        
        df = pd.read_csv(csv_path)
        if 'date' not in df.columns:
            continue
            
        for date_str in df['date']:
            records.append({
                "tile": tile,
                "date": date_str
            })
            
    df_manifest = pd.DataFrame(records)
    df_manifest['date_obj'] = pd.to_datetime(df_manifest['date'])
    
    # 3. Apply Temporal Splitting & The 14-Day Purge Gap
    # We want Train: <=2021, Val: 2022, Test: >=2023.
    # To prevent 14-day rolling rain features from overlapping, we add a 14-day deadzone.
    
    train_end = pd.to_datetime('2021-12-17') # Leaves 14 days before 2022
    val_start = pd.to_datetime('2022-01-01')
    val_end   = pd.to_datetime('2022-12-17') # Leaves 14 days before 2023
    test_start= pd.to_datetime('2023-01-01')

    def assign_split(row):
        # Rule 1: Spatial Holdout ALWAYS goes to Test (Geographic Generalization)
        if row['tile'] in spatial_holdout_tiles:
            return 'test_spatial'
            
        # Rule 2: Temporal Logic (Chronological Generalization)
        d = row['date_obj']
        if d <= train_end:
            return 'train'
        elif val_start <= d <= val_end:
            return 'val'
        elif d >= test_start:
            return 'test_temporal'
        else:
            return 'drop' # Falls inside the 14-day purge gap!

    df_manifest['split'] = df_manifest.apply(assign_split, axis=1)
    
    # Drop the deadzone events
    df_manifest = df_manifest[df_manifest['split'] != 'drop'].copy()
    
    # 4. Construct File Paths
    # IMPORTANT: Adjust these f-strings to match EXACTLY how your files are saved!
    
    def build_paths(row):
        tile = row['tile']
        d_str = row['date'].replace('-', '_') # "2020-04-15" -> "2020_04_15"
        
        # Static Layers
        dem = data_root / "Kenya_Flood_Dataset" / "SRTM" / f"{tile}_dem.tif"
        hand = data_root / "Kenya_Flood_Dataset" / "MERIT_Hydro" / f"{tile}_hand.tif"
        lc = data_root / "Kenya_Flood_Dataset" / "ESA_WorldCover" / f"{tile}_v200.tif"
        
        # Dynamic Layers
        rain = data_root / "Kenya_Flood_Dataset" / "CHIRPS" / f"{tile}_CHIRPS_precip_14d_sum_{d_str}.tif"
        
        # Target Mask (Replace this with wherever you save your final thresholded masks)
        target = data_root / "flood_masks" / f"{tile}_{d_str}.tif"
        
        return pd.Series({
            "dem_path": str(dem),
            "hand_path": str(hand),
            "landcover_path": str(lc),
            "rain_14d_path": str(rain),
            "target_path": str(target)
        })

    paths_df = df_manifest.apply(build_paths, axis=1)
    df_manifest = pd.concat([df_manifest, paths_df], axis=1)
    
    # 5. (Optional) Verify paths exist
    if verify_paths:
        print("Verifying paths exist on disk... (This may take a minute)")
        def check_exists(row):
            # Check a sample of columns
            return Path(row['target_path']).exists() and Path(row['rain_14d_path']).exists()
            
        df_manifest['is_valid'] = df_manifest.apply(check_exists, axis=1)
        missing = len(df_manifest[~df_manifest['is_valid']])
        if missing > 0:
            print(f"⚠️ WARNING: {missing} rows have missing files on disk!")
        
        df_manifest = df_manifest[df_manifest['is_valid']]
        df_manifest.drop(columns=['is_valid'], inplace=True)

    # Clean up and save
    df_manifest.drop(columns=['date_obj'], inplace=True)
    df_manifest.reset_index(drop=True, inplace=True)
    df_manifest.index.name = 'sample_id'
    
    output_path = data_root / output_csv
    df_manifest.to_csv(output_path)
    
    # 6. Print Statistics
    print("\nManifest Generation Complete!")
    print(f"Saved to: {output_path}")
    print("\n--- Split Statistics ---")
    print(df_manifest['split'].value_counts().to_string())
    print("-" * 24)

    return df_manifest

# ==========================================
# Run the script
# ==========================================
if __name__ == "__main__":
    DATA_DIRECTORY = "D:/flood_data" # Change this to your main data folder
    
    manifest = create_manifest(
        data_root=DATA_DIRECTORY,
        output_csv="manifest.csv",
        spatial_holdout_tiles=["N00E039"], # Pick a tile representing a diverse geography
        verify_paths=False # Set to True once all your downloads finish
    )

What this script does for you:

1.  The Spatial Holdout (test_spatial): It takes all events from N00E039 and
    permanently locks them in a test_spatial category, ignoring the dates. This
    guarantees you have a way to test if your model learns physics instead of
    memorizing coordinates.
2.  The Temporal Split (test_temporal): Events from the remaining 11 tiles are
    split chronologically. This tests if your model can predict future weather
    events on land it has seen before.
3.  The 14-Day Purge: It drops any dates that fall between Dec 17, 2021 and
    Dec 31, 2021. If a flood happened on Dec 30, the model would normally use
    rain data from Dec 16–30. By purging this, we guarantee the Validation set
    (starting Jan 1, 2022) uses completely untouched weather data.
4.  The Boilerplate: It stitches the N00E036 tile strings and 2020_04_15 date
    strings together into perfect absolute filepaths.

Next Steps:

1.  Double check the build_paths function inside the script. Ensure the folder
    names (like Kenya_Flood_Dataset/SRTM/) perfectly match the folders created
    by your Google Earth Engine script earlier.
2.  Run this script once.
3.  You will now have a manifest.csv. Pass this CSV into the
    ManifestFloodDataset code provided in the previous step, and your dataloader
    will work seamlessly!


In [ ]:
ERA5_EXPORT_KWARGS = {
    'folder': 'Kenya_Flood_Dataset/ERA5/',
    'crs': 'EPSG:4326',
    'crs_transform': [0.1, 0, -180.05, 0, -0.1, 90.05]
}